<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/4_1_Na%C3%AFve_Bayes_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part IV. Bayesian View of Linear Models

## Naïve Bayes Classifier

## Generalized Linear Models (GLM)

## Bayesian Linear Regression

## Bayesian Logistic Regression

## Latent Variable Models: EM Algorithm and GMM

### Наивный Байес: вероятностная постановка, предположение независимости и вывод правила классификации

Линейные модели строят разделяющую гиперплоскость, а деревья — кусочно-постоянные области. Наивный байесовский классификатор предлагает третий путь: оценить совместное распределение признаков и класса, а затем применить теорему Байеса. Его «наивность» — в предположении, что признаки условно независимы при заданном классе. Это допущение резко упрощает оценку и делает алгоритм быстрым, интерпретируемым и эффективным даже при высокой размерности.

#### 1. Теорема Байеса для классификации

Пусть $Y \in \{1,\dots,K\}$ — случайная величина, обозначающая класс, а $\mathbf{X} = (X_1,\dots,X_D)$ — вектор признаков, который может содержать как непрерывные, так и дискретные компоненты. Цель классификации — для нового наблюдения $\mathbf{x}$ предсказать наиболее вероятный класс $Y$, опираясь на известные вероятностные соотношения между классом и признаками.

Ключевым инструментом служит теорема Байеса, связывающая апостериорную вероятность класса с априорной вероятностью и функцией правдоподобия:

$$
\mathbb{P}(Y = k \mid \mathbf{X} = \mathbf{x}) = \frac{\mathbb{P}(Y = k) \, p(\mathbf{x} \mid Y = k)}{p(\mathbf{x})}, \quad k = 1,\dots,K. \tag{1}
$$

Здесь
- $\mathbb{P}(Y = k)$ — априорная вероятность класса $k$, отражающая его долю в генеральной совокупности;
- $p(\mathbf{x} \mid Y = k)$ — условная плотность (или вероятность, если $\mathbf{X}$ дискретен) вектора признаков при условии, что объект принадлежит классу $k$;
- $p(\mathbf{x}) = \sum_{k=1}^K \mathbb{P}(Y = k) \, p(\mathbf{x} \mid Y = k)$ — маргинальная плотность вектора признаков, обеспечивающая нормировку апостериорных вероятностей.

Поскольку знаменатель $p(\mathbf{x})$ не зависит от класса, оптимальное в смысле минимизации вероятности ошибки решающее правило (байесовский классификатор) принимает вид

$$
\hat{y} = \arg\max_{k} \; \mathbb{P}(Y = k \mid \mathbf{x}) = \arg\max_{k} \; \mathbb{P}(Y = k) \, p(\mathbf{x} \mid Y = k). \tag{2}
$$

Для вычислительного удобства часто переходят к логарифмам, превращая произведение в сумму:

$$
\hat{y} = \arg\max_{k} \; \bigl[ \log \mathbb{P}(Y = k) + \log p(\mathbf{x} \mid Y = k) \bigr]. \tag{3}
$$

Таким образом, задача классификации сводится к оцениванию априорных вероятностей классов и $D$-мерных условных распределений признаков. Именно здесь возникает главная сложность: оценка многомерной плотности в пространстве высокой размерности требует экспоненциально большого объёма данных — феномен, известный как «проклятие размерности».

#### 2. Предположение условной независимости (наивное)

Наивный байесовский классификатор обходит это препятствие с помощью радикального упрощающего предположения: **признаки $X_1,\dots,X_D$ условно независимы при заданном значении класса**. Формально это означает, что для каждого класса $k$

$$
p(\mathbf{x} \mid Y = k) = \prod_{j=1}^D p(x_j \mid Y = k). \tag{4}
$$

Подстановка факторизации (4) в решающее правило (3) приводит к **правилу наивного Байеса**:

$$
\hat{y} = \arg\max_{k} \; \Bigl[ \log \mathbb{P}(Y = k) + \sum_{j=1}^D \log p(x_j \mid Y = k) \Bigr]. \tag{5}
$$

Факторизация приносит колоссальное снижение сложности. Вместо того чтобы оценивать $K$ многомерных плотностей, каждая из которых требует работы с $D$-мерным пространством, достаточно оценить $K \times D$ одномерных распределений. Число параметров растёт линейно по $D$, а не экспоненциально. Именно это свойство делает наивный Байес практичным при $D \gg N$, где $N$ — число обучающих примеров.

Слово «наивный» отражает осознанное пренебрежение зависимостями между признаками. В реальных данных признаки практически всегда коррелированы. Например, в задаче классификации текстов слова «деньги» и «счёт» часто встречаются совместно в финансовых документах. Тем не менее наивный классификатор нередко показывает конкурентоспособное качество. Причина в том, что даже если оценка апостериорных вероятностей смещена из-за нарушения независимости, решающая граница может оставаться корректной: классификатору важен лишь порядок дискриминантных функций, а не их точные значения. Особенно хорошо это работает в разреженных данных, где каждая пара признаков имеет множество нулевых комбинаций, и агрегирование свидетельств нивелирует искажения.

#### 3. Структура модели и параметры

Полная вероятностная модель наивного Байеса состоит из априорного распределения классов и набора одномерных условных распределений каждого признака.

**Априорные вероятности классов** $\pi_k = \mathbb{P}(Y = k)$ обычно оцениваются относительными частотами в обучающей выборке:

$$
\hat{\pi}_k = \frac{N_k}{N}, \quad N_k = \sum_{i=1}^N \mathbf{1}[y_i = k],
$$

где $N$ — общее число примеров. При необходимости можно использовать сглаживание Лапласа или иные методы, чтобы избежать нулевых вероятностей.

**Условные распределения признаков.** Для каждого класса $k$ и каждого признака $j$ выбирается параметрическое семейство, отвечающее природе $X_j$:
- *Непрерывные признаки* — часто полагают $p(x_j \mid Y=k) = \mathcal{N}(x_j \mid \mu_{kj}, \sigma_{kj}^2)$.
- *Бинарные признаки* (наличие/отсутствие свойства) — распределение Бернулли $p(x_j \mid Y=k) = \theta_{kj}^{x_j} (1-\theta_{kj})^{1-x_j}$.
- *Счётные признаки* (частоты) — мультиномиальное или пуассоновское распределение.

Параметры каждого одномерного распределения оцениваются независимо по подвыборке соответствующего класса. Например, для гауссовского случая $\hat{\mu}_{kj}$ — выборочное среднее, $\hat{\sigma}_{kj}^2$ — выборочная дисперсия.

Общее число свободных параметров модели составляет $O(K D)$ — очень скромное даже для задач с тысячами признаков.

#### 4. Почему наивный Байес устойчив к размерности?

Классические методы классификации (например, логистическая регрессия без регуляризации или квадратичный дискриминантный анализ) требуют оценки ковариационной структуры, число параметров которой растёт как $O(D^2)$. При $D \gg N$ такие модели переобучаются или становятся вычислительно несостоятельными.

Наивный Байес радикально решает эту проблему благодаря факторизации (4). Все одномерные распределения оцениваются независимо, и число параметров линейно по $D$. Если каждое из этих распределений выбрано разумно (например, нормальное с небольшим числом параметров), то даже при $D \gg N$ можно получить состоятельные оценки — в каждом одномерном оценивании участвуют все наблюдения данного класса. Это даёт наивному Байесу уникальную способность работать в режимах, губительных для многих других алгоритмов.

Платой за устойчивость является потенциально сильное смещение апостериорных вероятностей при наличии существенных корреляций. Если признаки сильно зависимы, произведение маргиналов может значительно искажать форму совместной плотности. К счастью, часто классификация остаётся верной: ошибка смещения примерно одинакова для всех классов или не меняет ранжирования дискриминантных функций. Катастрофическое ухудшение наступает, когда структура корреляций радикально различается между классами. Например, если в одном классе пара признаков имеет высокую положительную корреляцию, а в другом — отрицательную, факторизация искажает решающую границу до неузнаваемости.



#### 5. Связь с линейным классификатором для некоторых семейств

Интересная и важная особенность наивного Байеса заключается в том, что для ряда параметрических семейств он порождает линейную разделяющую границу, тем самым сближаясь с линейными моделями, такими как логистическая регрессия и линейный дискриминантный анализ.

Предположим, что для каждого признака $j$ условное распределение $p(x_j \mid Y=k)$ принадлежит экспоненциальному семейству с параметром разброса, одинаковым для всех классов. Типичный пример — нормальные признаки с равными дисперсиями: $\mathcal{N}(x_j \mid \mu_{kj}, \sigma_j^2)$, где $\sigma_j^2$ не зависит от $k$. Тогда, подставляя произведение этих нормальных плотностей в логарифмическое решающее правило (5) и сокращая квадратичные члены, зависящие от класса, получаем, что дискриминантная функция является линейной по $\mathbf{x}$. Следовательно, поверхность, разделяющая любые два класса, — гиперплоскость.

**Углублённый взгляд: вывод линейной границы для двух классов.**  
Рассмотрим бинарный случай $Y \in \{1,2\}$ и предположим, что условные распределения признаков нормальны:

$$
p(x_j \mid Y = k) = \frac{1}{\sqrt{2\pi\sigma_j^2}} \exp\!\left(-\frac{(x_j - \mu_{kj})^2}{2\sigma_j^2}\right),
$$

где $\sigma_j^2$ одинаковы для $k=1,2$. Вычислим логарифм отношения апостериорных вероятностей:

$$
\log\frac{\mathbb{P}(Y=1 \mid \mathbf{x})}{\mathbb{P}(Y=2 \mid \mathbf{x})} =
\log\frac{\pi_1}{\pi_2} + \sum_{j=1}^D \log\frac{p(x_j \mid Y=1)}{p(x_j \mid Y=2)}.
$$

Для каждого $j$:

$$
\log\frac{p(x_j \mid Y=1)}{p(x_j \mid Y=2)} =
-\frac{(x_j - \mu_{1j})^2}{2\sigma_j^2} + \frac{(x_j - \mu_{2j})^2}{2\sigma_j^2}
= \frac{\mu_{1j} - \mu_{2j}}{\sigma_j^2} x_j - \frac{\mu_{1j}^2 - \mu_{2j}^2}{2\sigma_j^2}.
$$

Суммируя по всем признакам и добавляя логарифм отношения априорных вероятностей, получаем

$$
\log\frac{\mathbb{P}(Y=1 \mid \mathbf{x})}{\mathbb{P}(Y=2 \mid \mathbf{x})} =
\sum_{j=1}^D \frac{\mu_{1j} - \mu_{2j}}{\sigma_j^2} \, x_j \;-\;
\frac{1}{2}\sum_{j=1}^D \frac{\mu_{1j}^2 - \mu_{2j}^2}{\sigma_j^2} \;+\; \log\frac{\pi_1}{\pi_2}.
$$

Это выражение имеет вид $\mathbf{w}^T \mathbf{x} + b$, где

$$
w_j = \frac{\mu_{1j} - \mu_{2j}}{\sigma_j^2}, \qquad
b = -\frac{1}{2}\sum_{j=1}^D \frac{\mu_{1j}^2 - \mu_{2j}^2}{\sigma_j^2} + \log\frac{\pi_1}{\pi_2}.
$$

Следовательно, граница между классами $\mathbf{w}^T \mathbf{x} + b = 0$ является гиперплоскостью. При равных априорных вероятностях $\pi_1 = \pi_2$ коэффициент $b$ принимает вид $-\frac{1}{2}(\boldsymbol{\mu}_1 - \boldsymbol{\mu}_2)^T \boldsymbol{\Sigma}^{-1}(\boldsymbol{\mu}_1 + \boldsymbol{\mu}_2)$, где $\boldsymbol{\Sigma} = \operatorname{diag}(\sigma_1^2, \dots, \sigma_D^2)$, и классификатор совпадает с линейным дискриминантом Фишера. Аналогичный результат справедлив и для других экспоненциальных семейств с общим параметром разброса (мультиномиальное, пуассоновское), что объясняет успех наивного Байеса в задачах типа классификации текстов, где он неявно действует как линейная модель.




**Бернуллиевские признаки.** Если признаки бинарны, $x_j \in \{0,1\}$, и $p(x_j \mid Y=k) = \theta_{kj}^{x_j}(1-\theta_{kj})^{1-x_j}$, то логарифм отношения правдоподобий принимает вид

$$
\log\frac{p(x_j \mid Y=1)}{p(x_j \mid Y=2)} =
x_j \log\frac{\theta_{1j}(1-\theta_{2j})}{\theta_{2j}(1-\theta_{1j})} +
\log\frac{1-\theta_{1j}}{1-\theta_{2j}}.
$$

Суммируя по $j$ и добавляя $\log(\pi_1/\pi_2)$, снова получаем линейную функцию $\mathbf{w}^T\mathbf{x} + b$, где

$$
w_j = \log\frac{\theta_{1j}(1-\theta_{2j})}{\theta_{2j}(1-\theta_{1j})}, \qquad
b = \sum_{j=1}^D \log\frac{1-\theta_{1j}}{1-\theta_{2j}} + \log\frac{\pi_1}{\pi_2}.
$$

Это объясняет, почему наивный Байес для текстов (MultinomialNB, BernoulliNB) эквивалентен линейному классификатору с весами, зависящими от частот слов в классах.



In [ ]:
# @title Наивный Байес vs Логистическая регрессия: влияние корреляции признаков
import numpy as np
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

# Данные с коррелированными признаками
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, random_state=42)
# Добавляем корреляцию вручную
X = X @ np.array([[1, 0.8], [0.8, 1]])  # корреляция ~0.8

# Модели
nb = GaussianNB().fit(X, y)
lr = LogisticRegression().fit(X, y)

# Сетка для визуализации
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200),
                     np.linspace(X[:,1].min()-1, X[:,1].max()+1, 200))
Z_nb = nb.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
Z_lr = lr.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.contourf(xx, yy, Z_nb, alpha=0.3, cmap='bwr')
ax1.scatter(X[:,0], X[:,1], c=y, cmap='bwr', edgecolors='k')
ax1.set_title(f'Наивный Байес (GaussianNB)\nAccuracy: {nb.score(X,y):.3f}')

ax2.contourf(xx, yy, Z_lr, alpha=0.3, cmap='bwr')
ax2.scatter(X[:,0], X[:,1], c=y, cmap='bwr', edgecolors='k')
ax2.set_title(f'Логистическая регрессия\nAccuracy: {lr.score(X,y):.3f}')

for ax in (ax1, ax2):
    ax.set_xlabel('Признак 1'); ax.set_ylabel('Признак 2')
plt.tight_layout()
plt.show()
print("Обратите внимание: граница Наивного Байеса перпендикулярна осям (из-за независимости),")
print("а Логистическая регрессия строит диагональную границу, учитывая корреляцию.")
print("Несмотря на смещённую границу, точность Наивного Байеса может быть высокой.")



#### Вопросы для самопроверки

**Для начинающих**
1. В чём заключается «наивное» предположение наивного байесовского классификатора и для чего оно вводится?
2. Как теорема Байеса используется для предсказания метки класса?
3. Почему модель легко масштабируется на задачи с очень большим числом признаков?
4. Как обычно оцениваются априорные вероятности классов $\pi_k$?
5. Что означает утверждение «признаки условно независимы при заданном классе»? Приведите простой пример.

**Для профессионалов**
1. Докажите, что если условные распределения признаков принадлежат экспоненциальному семейству с равными по классам дисперсиями, то граница, порождаемая наивным Байесом, является линейной.
2. Объясните, почему наивный байесовский классификатор может давать правильную классификацию даже при заметных нарушениях предположения независимости. При каких условиях это преимущество исчезает?
3. Выведите выражение для логарифма отношения апостериорных вероятностей двух классов в общем виде и конкретизируйте его для случая бернуллиевских (бинарных) признаков.
4. Приведите пример (из анализа текстов, медицины или другой области), когда сильная корреляция признаков приводит к катастрофическому смещению апостериорных вероятностей в наивном Байесе и полностью разрушает качество классификации.

### Оценка параметров: метод максимального правдоподобия для разных типов признаков и сглаживание Лапласа

После того как вероятностная структура наивного байесовского классификатора зафиксирована, следующий шаг — оценка параметров модели по обучающей выборке. В наивном Байесе, благодаря предположению условной независимости и разделению априорных вероятностей от условных распределений признаков, задача распадается на множество независимых одномерных оцениваний. Это не только упрощает вычисления, но и делает модель особенно прозрачной. Классический подход — метод максимального правдоподобия (MLE), однако при работе с дискретными признаками неизбежно возникает проблема нулевых вероятностей, для решения которой применяется сглаживание Лапласа, имеющее изящную байесовскую интерпретацию.

#### 1. Метод максимального правдоподобия для параметров наивного Байеса

Пусть обучающая выборка состоит из $n$ независимых пар $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$, где $y_i \in \{1,\dots,K\}$ и $\mathbf{x}_i \in \mathbb{R}^D$ (или дискретное пространство). Полное правдоподобие, согласно факторизованной модели, имеет вид

$$
\mathcal{L}(\theta) = \prod_{i=1}^n \left[ \mathbb{P}(Y = y_i) \prod_{j=1}^D p(x_{ij} \mid Y = y_i) \right].
$$

Переходя к логарифмическому правдоподобию, получаем сумму двух независимых частей:

$$
\ell(\theta) = \underbrace{\sum_{i=1}^n \log \mathbb{P}(Y = y_i)}_{\text{априорные вероятности}} \;+\; \underbrace{\sum_{i=1}^n \sum_{j=1}^D \log p(x_{ij} \mid Y = y_i)}_{\text{условные распределения признаков}}. \tag{1}
$$

Благодаря аддитивной структуре, задача максимизации $\ell(\theta)$ распадается: априорные вероятности классов $\pi_k = \mathbb{P}(Y=k)$ и параметры каждого условного распределения $p(x_j \mid Y=k)$ можно оценивать раздельно. Априорные вероятности оцениваются как относительные частоты классов:

$$
\hat{\pi}_k = \frac{n_k}{n}, \quad n_k = \sum_{i=1}^n \mathbf{1}[y_i = k]. \tag{2}
$$

Параметры распределения признака $j$ для класса $k$ оцениваются только по тем примерам, для которых $y_i = k$. Таким образом, задача сводится к $K \times D$ независимым одномерным задачам оценивания, каждая из которых решается стандартными методами MLE для соответствующего параметрического семейства.

#### 2. Гауссовский наивный Байес (GaussianNB)

Для непрерывных признаков наиболее распространён выбор нормального распределения:

$$
p(x_j \mid Y = k) = \mathcal{N}\!\bigl(x_j \;\big|\; \mu_{kj}, \sigma_{kj}^2\bigr) = \frac{1}{\sqrt{2\pi\sigma_{kj}^2}} \exp\!\left( -\frac{(x_j - \mu_{kj})^2}{2\sigma_{kj}^2} \right). \tag{3}
$$

Логарифм правдоподобия для признака $j$ и класса $k$ с точностью до констант имеет вид

$$
\ell_{kj}(\mu_{kj}, \sigma_{kj}^2) = -\frac{n_k}{2} \log \sigma_{kj}^2 - \frac{1}{2\sigma_{kj}^2} \sum_{i: y_i=k} (x_{ij} - \mu_{kj})^2.
$$

Максимизация даёт классические MLE-оценки:

$$
\hat{\mu}_{kj} = \frac{1}{n_k} \sum_{i: y_i=k} x_{ij}, \qquad
\hat{\sigma}_{kj}^2 = \frac{1}{n_k} \sum_{i: y_i=k} (x_{ij} - \hat{\mu}_{kj})^2. \tag{4}
$$

Следует отметить, что оценка дисперсии по формуле (4) является смещённой (деление на $n_k$, а не на $n_k-1$). На практике, особенно при малых $n_k$, иногда применяют несмещённую оценку, однако для целей классификации эта разница, как правило, несущественна. Важнее численная устойчивость: к оценённой дисперсии часто добавляют малую константу $\epsilon \sim 10^{-9}$, чтобы избежать вырожденных или нулевых значений, которые могут привести к бесконечным логарифмам правдоподобия.


Важнее численная устойчивость: если дисперсия оказывается нулевой (например, для вырожденного признака), в знаменателе экспоненты и под логарифмом возникает деление на ноль, что превращает правдоподобие в бесконечность или NaN. Поэтому к оценённой дисперсии часто добавляют малую константу $\epsilon \sim 10^{-9}$, гарантируя $\hat{\sigma}_{kj}^2 > 0$.




#### 3. Мультиномиальный наивный Байес (MultinomialNB)

Когда признаки представляют собой счётчики (например, сколько раз слово встретилось в документе), естественной моделью является мультиномиальное распределение. В классической постановке «мешок слов» каждый документ класса $k$ рассматривается как последовательность независимых событий, порождаемых общим для класса дискретным распределением на словаре размера $D$. Параметр $\theta_{kj}$ интерпретируется как вероятность появления слова $j$ в одном испытании при условии класса $k$, причём $\sum_{j=1}^D \theta_{kj} = 1$.

Мультиномиальное правдоподобие для всех документов класса $k$ пропорционально

$$
\prod_{j=1}^D \theta_{kj}^{\; \sum_{i: y_i=k} x_{ij}}.
$$

Максимизация логарифма правдоподобия с учётом ограничения $\sum_{j=1}^D \theta_{kj} = 1$ (например, методом множителей Лагранжа) приводит к оценкам

$$
\hat{\theta}_{kj}^{\text{MLE}} = \frac{\sum_{i: y_i=k} x_{ij}}{\sum_{i: y_i=k} \sum_{j'=1}^D x_{ij'}}. \tag{5}
$$

Иными словами, $\hat{\theta}_{kj}$ есть доля вхождений слова $j$ среди всех слов в документах класса $k$. Однако на практике MLE-оценка (5) страдает от проблемы нулевых вероятностей: если какое-то слово не встречается в обучающих документах класса $k$, его оценённая вероятность равна нулю, и любой новый документ, содержащий это слово, получит нулевое правдоподобие для класса $k$. Ниже мы подробно обсудим решение этой проблемы — сглаживание Лапласа.

#### 4. Бернуллиевский наивный Байес (BernoulliNB)

В некоторых задачах признаки бинарны: $x_j \in \{0,1\}$, что соответствует наличию или отсутствию некоторого свойства (например, встречается ли слово в документе хотя бы один раз). Модель для каждого признака — распределение Бернулли:

$$
p(x_j \mid Y=k) = \theta_{kj}^{x_j} (1 - \theta_{kj})^{1 - x_j}. \tag{6}
$$

Параметр $\theta_{kj}$ — вероятность того, что признак $j$ равен 1 при условии класса $k$. Логарифмическое правдоподобие для класса $k$ и признака $j$:

$$
\ell_{kj}(\theta_{kj}) = \left(\sum_{i: y_i=k} x_{ij}\right) \log \theta_{kj} + \left(n_k - \sum_{i: y_i=k} x_{ij}\right) \log(1 - \theta_{kj}).
$$

Максимизация приводит к MLE-оценке:

$$
\hat{\theta}_{kj}^{\text{MLE}} = \frac{1}{n_k} \sum_{i: y_i=k} x_{ij}. \tag{7}
$$

Как и в мультиномиальном случае, MLE может давать нулевые или единичные вероятности, что снова требует регуляризации.

#### 5. Сглаживание Лапласа и проблема нулевых вероятностей

Проблема нулевых вероятностей — фундаментальная трудность, с которой сталкиваются все модели, основанные на подсчёте частот дискретных признаков. Если некоторое значение признака $j$ (или, в мультиномиальном случае, некоторое слово) ни разу не встретилось в обучающей выборке для класса $k$, то MLE-оценка его вероятности равна нулю. В силу факторизованной формы правдоподобия наивного Байеса единственный нулевой множитель обнуляет всё произведение, и класс $k$ никогда не будет предсказан для объектов, содержащих это значение признака, даже если все остальные признаки свидетельствуют в пользу этого класса. Такая жёсткость неоправданна и приводит к переобучению.

Решение заключается в добавлении **псевдосчёта** $\alpha > 0$ ко всем наблюдаемым частотам — приём, известный как сглаживание Лапласа (или аддитивное сглаживание). Для мультиномиального наивного Байеса оценка принимает вид

$$
\hat{\theta}_{kj} = \frac{\left(\sum_{i: y_i=k} x_{ij}\right) + \alpha}{\left(\sum_{i: y_i=k} \sum_{j'=1}^D x_{ij'}\right) + \alpha \cdot D}. \tag{8}
$$

Для бернуллиевской модели аналогично

$$
\hat{\theta}_{kj} = \frac{\left(\sum_{i: y_i=k} x_{ij}\right) + \alpha}{n_k + 2\alpha}. \tag{9}
$$

Параметр $\alpha$, обычно выбираемый равным $1$ (классическое сглаживание Лапласа), можно интерпретировать как количество фиктивных «наблюдений» каждого признака/слова, добавленных ко всем классам. Чем больше $\alpha$, тем сильнее оценки стягиваются к равномерному распределению, уменьшая дисперсию, но увеличивая смещение.

> **Углублённый взгляд: байесовская интерпретация сглаживания Лапласа**  
> Сглаживание Лапласа имеет строгое обоснование в рамках байесовского подхода. Рассмотрим мультиномиальную модель и введём априорное распределение на вектор параметров $\boldsymbol{\theta}_k = (\theta_{k1}, \dots, \theta_{kD})$ в виде распределения Дирихле:
> $$
> p(\boldsymbol{\theta}_k) = \operatorname{Dirichlet}(\boldsymbol{\theta}_k \mid \alpha, \dots, \alpha) \propto \prod_{j=1}^D \theta_{kj}^{\alpha - 1}.
> $$
> Параметр $\alpha$ управляет концентрацией распределения: $\alpha = 1$ соответствует равномерному распределению на симплексе. Апостериорное распределение, получаемое умножением априорного на мультиномиальное правдоподобие, также является дирихлевским:
> $$
> p(\boldsymbol{\theta}_k \mid \mathcal{D}) \propto \prod_{j=1}^D \theta_{kj}^{\; n_{kj} + \alpha - 1}, \quad n_{kj} = \sum_{i: y_i=k} x_{ij}.
> $$
> Среднее апостериорного распределения $\mathbb{E}[\theta_{kj} \mid \mathcal{D}]$ в точности равно оценке (8). Таким образом, сглаженная оценка MLE эквивалентна апостериорному среднему при дирихлевском априорном с параметром $\alpha$. Выбор $\alpha$ отражает силу априорного убеждения о равновероятности всех слов/признаков до наблюдения данных. Аналогичная интерпретация справедлива и для бернуллиевской модели с бета-распределением в качестве априорного.

Сглаживание Лапласа — простой, но эффективный механизм, обеспечивающий, чтобы ни одно событие не имело нулевой вероятности. Это не только улучшает обобщающую способность, но и сообщает модели разумное поведение в условиях неполноты данных.



Запустите ячейку ниже, чтобы увидеть, как параметр сглаживания $\alpha$ влияет на точность MultinomialNB. При $\alpha \to 0$ модель переобучается, при больших $\alpha$ — недообучается. Оптимум обычно лежит в районе $\alpha \approx 1$ (классическое сглаживание Лапласа).

In [ ]:
# @title Влияние α (сглаживания Лапласа) на качество MultinomialNB
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import matplotlib.pyplot as plt

# Загружаем два класса: sci.space и alt.atheism
cats = ['sci.space', 'alt.atheism']
data = fetch_20newsgroups(subset='all', categories=cats, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.3, random_state=42)

# Преобразуем в мешок слов
vectorizer = CountVectorizer(stop_words='english')
X_tr = vectorizer.fit_transform(X_train)
X_te = vectorizer.transform(X_test)

# Перебираем разные α
alphas = np.logspace(-2, 3, 20)
train_acc, test_acc = [], []
for alpha in alphas:
    model = MultinomialNB(alpha=alpha)
    model.fit(X_tr, y_train)
    train_acc.append(accuracy_score(y_train, model.predict(X_tr)))
    test_acc.append(accuracy_score(y_test, model.predict(X_te)))

plt.figure(figsize=(8,5))
plt.semilogx(alphas, train_acc, 'o-', label='Train', color='blue')
plt.semilogx(alphas, test_acc, 's-', label='Test', color='red')
best_idx = np.argmax(test_acc)
plt.axvline(alphas[best_idx], color='gray', linestyle='--', alpha=0.7,
            label=f'Best α = {alphas[best_idx]:.2f}')
plt.xlabel('α (сглаживание Лапласа)')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.title('Влияние сглаживания Лапласа на MultinomialNB')
plt.show()
print(f"Лучшая точность на тесте: {test_acc[best_idx]:.3f} при α = {alphas[best_idx]:.2f}")



#### Вопросы для самопроверки

**Для начинающих**
1. Как вычислить оценку среднего значения для гауссовского признака в рамках наивного Байеса?
2. Для чего применяется сглаживание Лапласа? Что произойдёт, если $\alpha = 0$?
3. В чём основное различие между мультиномиальным и бернуллиевским наивным Байесом при анализе текстов?
4. Что означает сокращение MLE? Как MLE применяется для оценки априорных вероятностей классов?
5. Почему дисперсию в GaussianNB часто оценивают смещённой, и что добавляют для численной устойчивости?

**Для профессионалов**
1. Выведите MLE-оценку дисперсии для GaussianNB и обсудите, является ли она смещённой. Как использование несмещённой оценки $\frac{1}{n_k-1}$ может повлиять на классификацию?
2. Докажите формально, что сглаживание Лапласа для мультиномиальной модели соответствует апостериорному среднему при дирихлевском априорном распределении с параметром $\alpha$.
3. Проанализируйте влияние параметра $\alpha$ на bias‑variance компромисс в наивном Байесе. Почему слишком большое $\alpha$ может ухудшить качество классификации?
4. Почему в мультиномиальной модели оценки $\theta_{kj}$ необходимо нормировать по полной длине документа (сумме всех счётчиков)? Как изменится интерпретация, если использовать частоты TF‑IDF вместо сырых счётчиков?

### Сравнение вариантов наивного Байеса, выбор распределений и работа с разными типами данных

Построив математический фундамент и освоив методы оценки параметров, мы переходим к практической стороне наивного байесовского классификатора: какой его вариант выбрать для конкретной задачи, как ведут себя разные предположения о распределениях, и каковы сильные и слабые стороны модели в сравнении с другими подходами. Несмотря на кажущуюся простоту, правильный выбор типа наивного Байеса и понимание его поведения на реальных данных имеют решающее значение для успешного применения.

#### 1. Когда какой вариант использовать?

Выбор параметрического семейства для условных распределений признаков $p(x_j \mid Y=k)$ определяется природой данных. Наивный Байес предоставляет гибкость: для каждого признака можно задать своё распределение, но на практике обычно используют единый тип для всех признаков.

**GaussianNB (гауссовский).** Этот вариант предназначен для непрерывных признаков, распределение которых в каждом классе приблизительно нормально. Типичные примеры — физические измерения (рост, вес, температура, давление), финансовые индикаторы, спектральные характеристики. Если данные далеки от нормальности, можно применить преобразования (логарифмирование, степенное преобразование Бокса—Кокса), чтобы приблизить их к гауссовскому виду. Главное преимущество — крайняя простота: для каждого класса и признака запоминаются всего два числа (среднее и дисперсия).

**MultinomialNB (мультиномиальный).** Используется для неотрицательных дискретных признаков, представляющих собой счётчики событий. Классическая область — анализ текстов в представлении «мешок слов», где каждый признак соответствует слову из словаря, а значение — количество вхождений в документ. Мультиномиальное распределение естественно учитывает, что сумма счётчиков по всем словам равна длине документа, и эта длина несёт информацию (более длинные документы дают больше свидетельств). Однако модель не требует строгой нормировки длины, поскольку нормировка априорно заложена в мультиномиальное правдоподобие через сумму вероятностей $\sum_j \theta_{kj}=1$. Важно понимать, что при обучении на сырых частотах модель автоматически учится и на длине документов; если же нормировать каждый документ к единичной сумме (например, TF‑IDF), то интерпретация мультиномиального распределения нарушается, и лучше применять другие методы, такие как логистическая регрессия.

**BernoulliNB (бернуллиевский).** Применяется к бинарным признакам $\{0,1\}$. В текстовом анализе такой подход моделирует не частоту слова, а сам факт его присутствия в документе. Для коротких текстов (сообщения в социальных сетях, заголовки, sms-сообщения) бинарная модель часто оказывается более устойчивой, чем мультиномиальная, потому что повторение слова в коротком сообщении не несёт столь сильного сигнала, как сам факт его наличия. Кроме того, BernoulliNB широко используется в задачах с бинарными характеристиками объектов (наличие/отсутствие симптомов, флаги конфигурации).

**Другие варианты.** Наивный Байес не ограничивается тремя основными семействами. Если признаки являются положительными вещественными величинами с длинным хвостом (например, времена между событиями), можно применить экспоненциальное распределение. Для счётчиков с редкими событиями подходит пуассоновское распределение. В принципе, любое одномерное распределение может быть встроено в структуру наивного Байеса — достаточно уметь оценивать его параметры по методу максимального правдоподобия (или с помощью байесовских оценок) и вычислять логарифм плотности для нового наблюдения. Это делает модель легко адаптируемой к гетерогенным данным.

#### 2. Поведение на синтетических и реальных данных

Несмотря на наивное предположение о независимости, на практике классификатор часто показывает высокое качество. Рассмотрим синтетический пример: данные сгенерированы из двух многомерных нормальных распределений с полными ковариационными матрицами (т.е. признаки внутри класса сильно коррелированы). Обучим GaussianNB, который игнорирует корреляции и оценивает только диагональные дисперсии. Во многих случаях решающая граница, построенная наивным Байесом, оказывается близка к границе, полученной полным линейным (LDA) или квадратичным (QDA) дискриминантным анализом. Причина — в сохранении правильного ранжирования классов: хотя апостериорные вероятности искажены, разница их логарифмов часто имеет правильный знак в широкой области пространства признаков.

На реальных текстовых задачах, таких как фильтрация спама или классификация новостей, MultinomialNB и BernoulliNB десятилетиями служили основными алгоритмами благодаря своей скорости и неплохому качеству. Они часто дают точность, сравнимую с начальными версиями более сложных моделей, что делает их отличным выбором для baseline-моделирования. Однако стоит помнить, что NB склонен выдавать чрезмерно уверенные предсказания: апостериорные вероятности часто лежат очень близко к 0 или 1, даже когда классификатор ошибается. Калибровка таких вероятностей обычно плоха, но для задач, где требуется лишь ранжирование (например, сортировка документов по релевантности), это не является критичным недостатком.

#### 3. Обработка пропущенных значений

Одно из приятных свойств наивного Байеса — естественная устойчивость к пропущенным данным. Поскольку все признаки обрабатываются независимо, пропуск значения $x_j$ для конкретного тестового объекта можно просто исключить из произведения:

$$
p(\mathbf{x} \mid Y = k) \propto \prod_{j: x_j \text{ не пропущен}} p(x_j \mid Y = k).
$$

Это эквивалентно маргинализации пропущенного признака: мы не добавляем свидетельства ни за, ни против класса по этому измерению. На этапе обучения пропущенные значения можно либо удалять, либо подставлять средние значения, либо, что более последовательно, применять EM-алгоритм (ожидания-максимизации). В рамках EM пропущенные признаки рассматриваются как латентные переменные, и на E-шаге вычисляются их ожидаемые достаточные статистики при текущих параметрах, а на M-шаге параметры обновляются. Такой подход позволяет использовать всю доступную информацию без смещения, хотя и повышает вычислительную сложность.

#### 4. Преимущества и ограничения

**Преимущества наивного Байеса:**
- *Высокая скорость.* Обучение сводится к одному проходу по данным, предсказание — к вычислению $K \times D$ логарифмов плотностей. Общая сложность — $O(KD)$ на объект.
- *Интерпретируемость.* Параметры модели имеют прямую вероятностную интерпретацию. В текстовой классификации можно легко получить наиболее характерные слова для каждого класса.
- *Низкая дисперсия.* Благодаря сильным предположениям модель имеет мало параметров и, следовательно, меньше подвержена переобучению при малых выборках.
- *Масштабируемость на высокие размерности.* За счёт факторизации число параметров растёт линейно с размерностью.
- *Работа с пропущенными данными и гетерогенными признаками.*

**Недостатки:**
- *Предположение независимости.* Реальные корреляции признаков игнорируются, что может приводить к смещённым оценкам апостериорных вероятностей и в некоторых случаях — к неверным решениям.
- *Неспособность улавливать взаимодействия.* Если решающая граница определяется сложными комбинациями признаков, наивный Байес может её принципиально не аппроксимировать.
- *Чувствительность к избыточным признакам.* Пара высококоррелированных признаков фактически удваивает свой вклад в решение, что может искажать классификацию. В таких случаях полезно предварительное отбор признаков или уменьшение размерности.

#### 5. Связь с другими моделями

Наивный байесовский классификатор занимает чёткое положение в спектре вероятностных моделей. Рассмотрим его отношение к дискриминантному анализу и логистической регрессии.

**Связь с LDA и QDA.** Если предположить, что признаки внутри каждого класса имеют многомерное нормальное распределение, но наивный Байес дополнительно предполагает диагональность ковариационной матрицы, то он становится частным случаем квадратичного дискриминантного анализа (QDA) с диагональными ковариациями. Если же ещё и все диагональные элементы (дисперсии) равны между классами, получаем линейный дискриминантный анализ (LDA) с диагональной ковариационной матрицей. Таким образом, GaussianNB есть LDA/QDA с нулевыми внедиагональными элементами. Эта связь объясняет, почему при приблизительно нормальных данных и умеренных корреляциях NB может давать разумные результаты: он всё равно оценивает правильное направление разделяющей гиперплоскости.

**Связь с логистической регрессией.** Как было показано в разделе о линейности границы, для экспоненциального семейства с равными дисперсиями NB порождает линейную границу. Логистическая регрессия (LR) также строит линейную границу, но принципиально иным способом. NB — **генеративная** модель: она описывает совместное распределение $p(\mathbf{x}, y)$ и затем применяет правило Байеса. LR — **дискриминативная**: она напрямую моделирует $p(y \mid \mathbf{x})$, минуя описание распределения признаков.

> **Углублённый взгляд: сравнение с логистической регрессией**  
> Различие между генеративным (NB) и дискриминативным (LR) подходами имеет глубокие последствия. Генеративная модель делает более сильные предположения о данных (нормальность, независимость), и если эти предположения выполняются, она асимптотически эффективна: требует меньше данных для достижения той же точности, чем дискриминативная модель. В случае нарушений предположений дискриминативная модель, не тратя параметры на описание $p(\mathbf{x})$, может сосредоточиться на границе и дать лучшую классификацию. На практике логистическая регрессия обычно выигрывает при больших выборках и сильных корреляциях признаков; NB — при малых выборках и высокой размерности, особенно в разреженных текстовых данных.  
> Ещё одно важное различие — калибровка вероятностей. Если предположения NB верны, его апостериорные вероятности хорошо откалиброваны. В противном случае они могут быть сильно смещены. Логистическая регрессия, минимизируя кросс-энтропию, обычно даёт лучше откалиброванные вероятности даже при нарушении предположений.  
> Формально, можно показать, что при определённых условиях NB и LR имеют одинаковую параметризацию границы, но веса NB оцениваются аналитически (из средних и дисперсий), а веса LR — итерационной оптимизацией функции потерь. Это приводит к разным bias-variance свойствам: NB имеет более высокое смещение (из-за предположения независимости), но низкую дисперсию, тогда как LR обладает меньшим смещением, но большей дисперсией.

Таким образом, наивный Байес остаётся незаменимым инструментом начального анализа данных, базовой модели и задач с жёсткими ограничениями по времени или памяти, несмотря на свою кажущуюся наивность.



Запустите ячейку ниже. На synthetic data с коррелированными признаками сравниваются границы решений GaussianNB, LDA и QDA. Хорошо видно, что NB строит границу, параллельную осям (следствие независимости), LDA — прямую диагональную границу, а QDA — криволинейную.

In [ ]:
# @title Сравнение GaussianNB, LDA и QDA
import numpy as np
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.datasets import make_classification

# Данные с коррелированными признаками
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, random_state=42)
X = X @ np.array([[1, 0.7], [0.7, 1]])  # добавляем корреляцию

models = {
    'GaussianNB': GaussianNB(),
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis()
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200),
                     np.linspace(X[:,1].min()-1, X[:,1].max()+1, 200))

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X, y)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X[:,0], X[:,1], c=y, cmap='bwr', edgecolors='k', s=20)
    ax.set_title(f'{name}\nAccuracy: {model.score(X,y):.3f}')
    ax.set_xlabel('Признак 1'); ax.set_ylabel('Признак 2')

plt.tight_layout()
plt.show()



#### Вопросы для самопроверки

**Для начинающих**
1. Для каких типов данных предназначены GaussianNB, MultinomialNB и BernoulliNB? Приведите по одному примеру для каждого.
2. Почему наивный байесовский классификатор может давать верные предсказания, даже если его предположение о независимости нарушено?
3. Как обрабатываются пропущенные значения в наивном Байесе при предсказании нового объекта?
4. В чём разница между генеративным и дискриминативным подходом к классификации на примере NB и логистической регрессии?

**Для профессионалов**
1. Выведите, что GaussianNB с равными по классам дисперсиями эквивалентен линейному дискриминантному анализу с диагональной ковариационной матрицей. Как изменится граница, если дисперсии разные?
2. Объясните, почему наивный Байес склонен давать чрезмерно уверенные предсказания (вероятности, близкие к 0 или 1) при нарушении предположения о независимости. Как это влияет на калибровку?
3. Сравните bias-variance компромисс наивного Байеса и логистической регрессии. При каких условиях NB будет иметь преимущество, несмотря на большее смещение?
4. Приведите конкретный пример (например, из медицинской диагностики), где корреляция признаков приводит к катастрофическому смещению в NB, и объясните, как логистическая регрессия или другая модель могла бы исправить ситуацию.

### Реализация наивного байесовского классификатора с нуля на Python

Рассмотренные теоретические принципы непосредственно воплощаются в программном коде. Ниже мы последовательно реализуем три основных варианта наивного Байеса — гауссовский, мультиномиальный и бернуллиевский — и сравним их с библиотечными аналогами из scikit-learn.

Запустите первую ячейку — она содержит все три класса и общие импорты.

In [ ]:
# @title Реализация GaussianNB, MultinomialNB и BernoulliNB с нуля
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# ============================================================
# GaussianNB
# ============================================================
class GaussianNB:
    def __init__(self, eps=1e-9):
        self.eps = eps
        self.classes_ = None
        self.priors_ = None
        self.mu_ = None
        self.sigma_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        self.priors_ = np.zeros(n_classes)
        self.mu_, self.sigma_ = [], []
        for idx, k in enumerate(self.classes_):
            X_k = X[y == k]
            n_k = X_k.shape[0]
            self.priors_[idx] = n_k / X.shape[0]
            self.mu_.append(np.mean(X_k, axis=0))
            self.sigma_.append(np.var(X_k, axis=0) + self.eps)
        self.priors_ = np.log(self.priors_)
        return self

    def _joint_log_likelihood(self, X):
        n_samples, n_classes = X.shape[0], len(self.classes_)
        log_likelihood = np.zeros((n_samples, n_classes))
        for idx in range(n_classes):
            mu, var = self.mu_[idx], self.sigma_[idx]
            log_p = -0.5 * np.sum(np.log(2 * np.pi * var)) \
                    - 0.5 * np.sum((X - mu) ** 2 / var, axis=1)
            log_likelihood[:, idx] = self.priors_[idx] + log_p
        return log_likelihood

    def predict(self, X):
        jll = self._joint_log_likelihood(X)
        return self.classes_[np.argmax(jll, axis=1)]

    def predict_proba(self, X):
        jll = self._joint_log_likelihood(X)
        jll -= np.max(jll, axis=1, keepdims=True)
        prob = np.exp(jll)
        prob /= np.sum(prob, axis=1, keepdims=True)
        return prob

# ============================================================
# MultinomialNB
# ============================================================
class MultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.priors_ = None
        self.log_theta_ = None
        self.n_features_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_samples, self.n_features_ = X.shape
        self.priors_ = np.zeros(n_classes)
        feature_count = np.zeros((n_classes, self.n_features_))
        class_count = np.zeros(n_classes)
        for idx, k in enumerate(self.classes_):
            X_k = X[y == k]
            class_count[idx] = X_k.sum()
            feature_count[idx, :] = X_k.sum(axis=0)
            self.priors_[idx] = X_k.shape[0] / n_samples
        theta = (feature_count + self.alpha) / (
            class_count[:, np.newaxis] + self.alpha * self.n_features_)
        self.log_theta_ = np.log(theta)
        self.priors_ = np.log(self.priors_)
        return self

    def _joint_log_likelihood(self, X):
        return self.priors_[np.newaxis, :] + X @ self.log_theta_.T

    def predict(self, X):
        return self.classes_[np.argmax(self._joint_log_likelihood(X), axis=1)]

    def predict_proba(self, X):
        jll = self._joint_log_likelihood(X)
        jll -= np.max(jll, axis=1, keepdims=True)
        prob = np.exp(jll)
        prob /= np.sum(prob, axis=1, keepdims=True)
        return prob

# ============================================================
# BernoulliNB
# ============================================================
class BernoulliNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.priors_ = None
        self.log_theta_ = None
        self.log_one_minus_theta_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_samples = X.shape[0]
        self.priors_ = np.zeros(n_classes)
        theta = np.zeros((n_classes, X.shape[1]))
        for idx, k in enumerate(self.classes_):
            X_k = X[y == k]
            n_k = X_k.shape[0]
            self.priors_[idx] = n_k / n_samples
            theta[idx, :] = (X_k.sum(axis=0) + self.alpha) / (n_k + 2 * self.alpha)
        self.log_theta_ = np.log(theta)
        self.log_one_minus_theta_ = np.log(1 - theta)
        self.priors_ = np.log(self.priors_)
        return self

    def _joint_log_likelihood(self, X):
        return (self.priors_[np.newaxis, :] +
                X @ self.log_theta_.T +
                (1 - X) @ self.log_one_minus_theta_.T)

    def predict(self, X):
        return self.classes_[np.argmax(self._joint_log_likelihood(X), axis=1)]

    def predict_proba(self, X):
        jll = self._joint_log_likelihood(X)
        jll -= np.max(jll, axis=1, keepdims=True)
        prob = np.exp(jll)
        prob /= np.sum(prob, axis=1, keepdims=True)
        return prob

print("Классы GaussianNB, MultinomialNB, BernoulliNB загружены.")

Теперь сравним наши реализации с библиотечными аналогами из scikit-learn на подходящих наборах данных: `iris` для GaussianNB, `digits` для MultinomialNB и бинаризованный `digits` для BernoulliNB.

In [ ]:
# @title Сравнение со sklearn
from sklearn.naive_bayes import GaussianNB as SklearnGaussianNB
from sklearn.naive_bayes import MultinomialNB as SklearnMultinomialNB
from sklearn.naive_bayes import BernoulliNB as SklearnBernoulliNB

# --- GaussianNB ---
iris = load_iris()
X_tr, X_te, y_tr, y_te = train_test_split(iris.data, iris.target, test_size=0.3, random_state=42)
gnb_custom = GaussianNB().fit(X_tr, y_tr)
gnb_sk = SklearnGaussianNB().fit(X_tr, y_tr)
print(f"GaussianNB custom: {accuracy_score(y_te, gnb_custom.predict(X_te)):.4f}")
print(f"GaussianNB sklearn: {accuracy_score(y_te, gnb_sk.predict(X_te)):.4f}")

# --- MultinomialNB ---
digits = load_digits()
X_d, y_d = digits.data, digits.target
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_d, y_d, test_size=0.3, random_state=0)
mnb_custom = MultinomialNB(alpha=1.0).fit(X_tr_d, y_tr_d)
mnb_sk = SklearnMultinomialNB(alpha=1.0).fit(X_tr_d, y_tr_d)
print(f"\nMultinomialNB custom: {accuracy_score(y_te_d, mnb_custom.predict(X_te_d)):.4f}")
print(f"MultinomialNB sklearn: {accuracy_score(y_te_d, mnb_sk.predict(X_te_d)):.4f}")

# --- BernoulliNB ---
X_d_bin = (X_d > 8).astype(int)
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_d_bin, y_d, test_size=0.3, random_state=0)
bnb_custom = BernoulliNB(alpha=1.0).fit(X_tr_b, y_tr_b)
bnb_sk = SklearnBernoulliNB(alpha=1.0).fit(X_tr_b, y_tr_b)
print(f"\nBernoulliNB custom: {accuracy_score(y_te_b, bnb_custom.predict(X_te_b)):.4f}")
print(f"BernoulliNB sklearn: {accuracy_score(y_te_b, bnb_sk.predict(X_te_b)):.4f}")

Наконец, исследуем влияние параметра сглаживания $\alpha$ на точность MultinomialNB. При $\alpha \to 0$ модель переобучается, при больших $\alpha$ — недообучается.

In [ ]:
# @title Влияние α на MultinomialNB
alphas = np.logspace(-2, 3, 30)
scores = []
for a in alphas:
    mnb = MultinomialNB(alpha=a).fit(X_tr_d, y_tr_d)
    scores.append(accuracy_score(y_te_d, mnb.predict(X_te_d)))

best_alpha = alphas[np.argmax(scores)]
print(f"Лучшая α: {best_alpha:.3f}, точность: {max(scores):.4f}")

plt.figure(figsize=(8, 5))
plt.semilogx(alphas, scores, 'b-o')
plt.axvline(best_alpha, color='r', linestyle='--', label=f'Оптимум α={best_alpha:.3f}')
plt.xlabel('α'); plt.ylabel('Accuracy')
plt.title('Зависимость точности MultinomialNB от сглаживания α')
plt.legend(); plt.grid(True)
plt.show()

Реализованные с нуля классы демонстрируют основные принципы наивного байесовского классификатора: факторизацию, логарифмирование и аддитивное сглаживание. Векторизованные вычисления обеспечивают высокую скорость, сопоставимую с промышленными библиотеками. Полученный код легко адаптировать под другие одномерные распределения, что делает наивный Байес одной из самых удобных моделей для быстрого прототипирования и обучения на разреженных данных.


При $\alpha \to 0$ могут появляться нулевые оценки вероятностей для отсутствовавших в обучении значений признаков, что обнуляет правдоподобие целых тестовых примеров и ухудшает точность. При чрезмерно больших $\alpha$ параметры стягиваются к равномерному распределению, и модель теряет различающую способность. Оптимальное значение, как правило, лежит в области $[0.1, 10]$.

#### 6. Заключительные замечания

Реализованные с нуля классы демонстрируют основные принципы наивного байесовского классификатора: факторизацию, логарифмирование и аддитивное сглаживание. Векторизованные вычисления обеспечивают высокую скорость, сопоставимую с промышленными библиотеками. Полученный код легко адаптировать под другие одномерные распределения, что делает наивный Байес одной из самых удобных моделей для быстрого прототипирования и обучения на разреженных данных.

---

#### Вопросы для самопроверки

**Для начинающих**
1. Как в классе `GaussianNB` вычисляется совместный логарифмический правдоподобие для одного объекта? Почему используется логарифмическая шкала?
2. Зачем в методе `fit` гауссовской модели к дисперсии добавляется небольшая константа `eps`?
3. Какие меры предприняты в `predict_proba` для предотвращения численного переполнения при вычислении экспоненты?
4. Что произойдёт при классификации мультиномиальным классификатором, если тестовый пример содержит признак со значением, не встречавшимся в обучении, и $\alpha = 0$? Как сглаживание решает эту проблему?

**Для профессионалов**
1. В классе `GaussianNB` вычисление логарифма правдоподобия реализовано без циклов по признакам. Объясните, за счёт каких операций NumPy достигается векторизация, и какова сложность полученного вычисления.
2. Почему при использовании `MultinomialNB` не следует предварительно нормировать признаки на длину документа? Как изменится вероятностная интерпретация модели в случае такой нормировки?
3. Сравните численную устойчивость вычисления `exp(log_prior + log_likelihood)` для получения вероятностей классов. Какие проблемы могут возникнуть при больших по модулю значениях логарифмического правдоподобия, и как `predict_proba` их решает?
4. Можно ли реализовать `_joint_log_likelihood` в `GaussianNB` вообще без циклов по классам, используя трёхмерные тензоры? Опишите подход и его возможные ограничения по памяти.

### Анализ смещения, калибровка и улучшение наивного байесовского классификатора

Мы завершаем главу, посвящённую наивному Байесу, обсуждением его статистических свойств, методов коррекции вероятностей и подходов к ослаблению самого сильного предположения модели — условной независимости признаков. Эти темы позволяют понять, когда классификатор работает хорошо, а когда требует модификации, и служат мостом к более сложным вероятностным моделям.

#### 1. Bias‑variance анализ наивного Байеса

Разложение ожидаемой ошибки предсказания на смещение (bias) и дисперсию (variance) даёт глубокое понимание поведения модели. Наивный байесовский классификатор характеризуется:

- **Высоким смещением.** Предположение об условной независимости накладывает жёсткие ограничения на форму разделяющей границы. Если истинное распределение данных содержит сильные корреляции признаков, модель принципиально не способна в точности воспроизвести оптимальную байесовскую границу — она будет систематически ошибаться даже при бесконечном объёме данных.
- **Низкой дисперсией.** Благодаря крайне малому числу параметров ($O(KD)$ вместо $O(KD^2)$ в полных ковариационных моделях) оценки параметров обладают малой изменчивостью от выборки к выборке. Алгоритм не склонен переобучаться.

Это объясняет классическое наблюдение: наивный Байес часто превосходит более гибкие модели (включая логистическую регрессию) на малых выборках. Малое количество данных не позволяет оценить сложные зависимости, и тогда низкая дисперсия NB становится решающим преимуществом. С ростом объёма данных дискриминативные модели (например, логистическая регрессия), имеющие меньшее смещение, сокращают разрыв и в итоге обходят NB по точности.

#### 2. Калибровка вероятностей наивного Байеса

Наивный байесовский классификатор, будучи генеративной моделью с сильными предположениями, склонен выдавать чрезмерно уверенные вероятности: предсказанные $p(y=k \mid \mathbf{x})$ часто лежат вблизи $0$ или $1$, даже когда классификатор ошибается. Это означает плохую калибровку. Для задач, где важно не только ранжирование, но и абсолютное значение вероятности (оценка рисков, принятие решений с порогами), требуется постобработка выходов модели.

Два основных метода калибровки — **Platt scaling** (логистическая калибровка) и **изотоническая регрессия** — были детально рассмотрены в разделе 7.3. Здесь мы применим Platt scaling к наивному Байесу, используя `CalibratedClassifierCV` из scikit-learn. Построим калибровочные кривые до и после калибровки, сравнивая частоту положительного класса в бинах с предсказанной средней вероятностью.


In [ ]:
# @title Калибровка GaussianNB: Platt scaling (интерактивный)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from ipywidgets import interact, IntSlider

# Данные
X, y = make_classification(n_samples=2000, n_features=10, n_informative=5,
                           n_redundant=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Модели
gnb = GaussianNB().fit(X_train, y_train)
cal_gnb = CalibratedClassifierCV(gnb, method='sigmoid', cv=5).fit(X_train, y_train)
probs = gnb.predict_proba(X_test)[:, 1]
probs_cal = cal_gnb.predict_proba(X_test)[:, 1]

def plot_calibration(n_bins=10):
    plt.figure(figsize=(8, 6))
    for p, label, fmt in [(probs, 'Исходный NB', 's-'),
                           (probs_cal, 'NB + Platt scaling', 'o-')]:
        frac, mean_pred = calibration_curve(y_test, p, n_bins=n_bins, strategy='uniform')
        plt.plot(mean_pred, frac, fmt, label=label)
    plt.plot([0, 1], [0, 1], 'k:', label='Идеальная калибровка')
    plt.xlabel('Средняя предсказанная вероятность')
    plt.ylabel('Доля положительных примеров')
    plt.legend(); plt.grid(True)
    plt.title(f'Калибровочные кривые (бинов: {n_bins})')
    plt.show()

interact(plot_calibration, n_bins=IntSlider(value=10, min=5, max=30, step=1, description='Число бинов'));


График демонстрирует, как Platt scaling приближает кривую к диагонали, существенно улучшая калибровку. Аналогичного эффекта можно добиться и для MultinomialNB в текстовых задачах.

#### 3. Улучшение наивного Байеса: взвешивание признаков

Предположение о независимости можно смягчить, не отказываясь полностью от факторизованной формы, если ввести веса признаков $w_j \ge 0$:

$$
\log p(\mathbf{x} \mid Y=k) \propto \sum_{j=1}^D w_j \log p(x_j \mid Y=k). \tag{1}
$$

Эта модель называется **взвешенным наивным Байесом** (Weighted Naive Bayes). Веса регулируют относительную важность признаков: $w_j=1$ соответствует стандартному NB, $w_j > 1$ увеличивает влияние признака, $w_j < 1$ — уменьшает. Веса можно подбирать путём максимизации правдоподобия или минимизации логистической потери на обучающей выборке.

Интересно, что если $p(x_j \mid Y=k)$ — нормальные распределения с равными дисперсиями и $w_j$ подбираются путём минимизации кросс-энтропийной потери, модель становится эквивалентной логистической регрессии. Действительно, логарифм отношения правдоподобий с весами

$$
\sum_{j=1}^D w_j \log\frac{p(x_j \mid Y=1)}{p(x_j \mid Y=2)}
= \sum_{j=1}^D w_j \frac{\mu_{1j} - \mu_{2j}}{\sigma_j^2} x_j + \text{const}
$$

является линейной функцией с коэффициентами $\beta_j = w_j (\mu_{1j} - \mu_{2j}) / \sigma_j^2$. Если разрешить $w_j$ настраиваться произвольно, мы получим ровно ту же линейную модель, что и логистическая регрессия, но обученную в два этапа: сначала оценки $\mu_{kj}, \sigma_j^2$, затем оптимизация $w_j$ (или непосредственно коэффициентов $\beta_j$). На практике взвешенный NB может служить компромиссом: сохраняется скорость и частичная интерпретируемость, но добавляется гибкость.

#### 4. Модели с ослабленным предположением независимости

Полный отказ от независимости требует моделирования зависимостей между признаками. Два направления особенно примечательны.

**Averaged One-Dependence Estimators (AODE).** Вместо того чтобы считать все признаки независимыми, AODE строит множество наивных байесовских классификаторов, в каждом из которых один признак выступает в роли «родительского» — относительно него все остальные признаки условно независимы:

$$
p(\mathbf{x} \mid Y=k) = \frac{1}{D} \sum_{j=1}^D p(x_j \mid Y=k) \prod_{l \neq j} p(x_l \mid x_j, Y=k).
$$

Результирующее предсказание — среднее по всем таким моделям. AODE сохраняет вычислительную эффективность (сложность $O(K D^2)$) и заметно снижает смещение, улавливая парные зависимости.

**Tree Augmented Naive Bayes (TAN).** Более структурный подход заключается в построении дерева зависимостей между признаками в каждом классе. Структура дерева (максимального остовного дерева на основе условной взаимной информации) добавляет к наивному Байесу рёбра, так что каждый признак имеет не более одного родителя среди других признаков. Логарифмическое правдоподобие становится суммой по рёбрам дерева:

$$
\log p(\mathbf{x} \mid Y=k) \propto \sum_{j=1}^D \log p(x_j \mid x_{\text{pa}(j)}, Y=k),
$$

где $\text{pa}(j)$ — родитель признака $j$ в дереве. TAN — простейшая байесовская сеть, и она служит естественным мостом к более общим графовым моделям.

Обе модели улучшают качество классификации при наличии ощутимых корреляций, сохраняя многие преимущества байесовского подхода.

> **Углублённый взгляд: теоретическая граница ошибки при нарушении независимости**  
> Domingos и Pazzani (1997) показали, что даже при нарушении условной независимости наивный Байес может давать оптимальную классификацию, если выполнено более слабое условие: для каждой пары классов $k,l$ отношение правдоподобий
> $$
> \frac{p(\mathbf{x} \mid Y=k)}{p(\mathbf{x} \mid Y=l)}
> $$
> правильно упорядочивает наблюдения. Формально, если признаки не слишком сильно коррелируют, ошибка NB ограничена. Более точно, если условная взаимная информация между любой парой признаков при фиксированном классе мала, то апостериорные вероятности, даваемые NB, сохраняют правильный порядок. Это объясняет эмпирический успех NB в широком спектре задач.

#### 5. Итоговое резюме и практические рекомендации

Наивный байесовский классификатор — один из самых элегантных и одновременно практичных алгоритмов машинного обучения. Его фундамент — теорема Байеса и смелое предположение об условной независимости признаков — приводит к модели с уникальным сочетанием свойств: линейная сложность обучения и предсказания, высокая интерпретируемость, устойчивость к высокой размерности и малому объёму данных.

**Когда использовать наивный Байес:**
- Выборки малого и среднего размера, особенно при $D \gg N$.
- Текстовые задачи (MultinomialNB, BernoulliNB) — как сильный baseline.
- Необходима быстрая, прозрачная и легко внедряемая модель.
- Признаки приблизительно независимы (например, после отбора или применения преобразований).
- Требуется оценка значимости отдельных признаков (через вероятности слов, средние).

**Когда следует предпочесть другие модели:**
- Признаки сильно коррелированы, и эти корреляции различны в разных классах — логистическая регрессия или SVM обычно лучше.
- Важна точная калибровка вероятностей — NB можно откалибровать, но дискриминативные модели изначально дают лучшие оценки.
- Требуется моделирование сложных нелинейных взаимодействий — здесь нужны деревья, нейронные сети и т.д.

Изучив наивный Байес во всех подробностях — от вероятностной постановки и оценки параметров до калибровки и расширений, — мы подготовили почву для перехода к более общим обобщённым линейным моделям (GLM). Там линейный предиктор связывается с откликом через произвольную функцию связи, а распределение отклика может принадлежать любому экспоненциальному семейству, снимая ограничение нормальности, но сохраняя мощь линейной структуры.

---

#### Вопросы для самопроверки (по всей главе «Наивный байесовский классификатор»)

**Для начинающих**
1. Почему апостериорные вероятности наивного Байеса часто бывают плохо откалиброваны (слишком близки к 0 или 1)?
2. Что такое Platt scaling и как он применяется для улучшения калибровки NB?
3. Как взвешивание признаков (Weighted Naive Bayes) связано с логистической регрессией?
4. В чём преимущество наивного Байеса на выборках малого объёма по сравнению с более гибкими моделями?
5. Как модель AODE ослабляет предположение о независимости?
6. Объясните, почему наивный байесовский классификатор обладает низкой дисперсией. Как это свойство связано с числом его параметров?

**Для профессионалов**
1. Выведите условие, при котором стандартный наивный Байес (с гауссовскими признаками и равными дисперсиями) даёт в точности те же предсказания, что и логистическая регрессия. Как введение весов признаков обобщает эту связь?
2. Покажите, что взвешивание признаков не нарушает предположение об условной независимости в вероятностной модели. Является ли взвешенный NB всё ещё генеративной моделью?
3. Сравните методы калибровки вероятностей для наивного Байеса и метода опорных векторов (SVM). Почему SVM также требует калибровки, и какой метод предпочтителен?
4. Опишите алгоритм построения Tree Augmented Naive Bayes (TAN). Как вычисляется структура дерева и параметры модели?
5. Приведите формальное утверждение (Domingos & Pazzani, 1997) о том, что ошибка наивного Байеса ограничена при слабой корреляции признаков. Объясните, почему сохранение правильного порядка апостериорных вероятностей может быть достаточным для верной классификации.